# 2. Deep Agents: a richer agent harness

Notebook 1 assembled the pieces ourselves.

This notebook keeps the same **Manager, Analyst and Reviewer** idea but uses **Deep Agents**.

We demonstrate native subagents, built-in file tools, shell execution, optional task planning and parallel delegation.

## 1. Imports and setup

**Important:** `LocalShellBackend` is convenient for a classroom demo but runs shell commands on the local machine. It is not a security sandbox.

In [1]:
from pathlib import Path
import re

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

MODEL = "gpt-5.6-luna"
# max_retries raised from the langchain_openai default of 2: the OpenAI SDK already
# retries rate-limit errors with exponential backoff built in, it just gives up too
# soon by default. This matters more here than in a single-agent notebook because
# parallel subagent delegation (section 10) can fire several LLM calls at once.
model = ChatOpenAI(model=MODEL, use_responses_api=True, max_retries=5)
search_model = ChatOpenAI(model=MODEL, use_responses_api=True, max_retries=5)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 2. Recreate the small local knowledge base

In [2]:
documents = [
    Document(
        page_content=(
            "Motor claims inflation rose because repair labour, replacement parts, "
            "vehicle technology and hire-car costs became more expensive. "
            "The insurer responded by increasing pricing and tightening claims controls."
        ),
        metadata={"source": "motor_claims_note.txt"},
    ),
    Document(
        page_content=(
            "A household insurer is testing generative AI to summarise claim notes. "
            "The pilot is intended to reduce administrative work, but human claims handlers "
            "remain responsible for coverage decisions and settlement authority."
        ),
        metadata={"source": "claims_ai_pilot.txt"},
    ),
    Document(
        page_content=(
            "Fraud teams combine rules, anomaly detection and investigator judgement. "
            "An AI assistant may help investigators search past cases, but false positives "
            "can create unnecessary referrals and customer friction."
        ),
        metadata={"source": "fraud_note.txt"},
    ),
    Document(
        page_content=(
            "Personally identifiable information in insurance can include customer names, "
            "email addresses, policy numbers, payment details and claim identifiers. "
            "Sensitive information should be minimised before it is sent to external systems."
        ),
        metadata={"source": "privacy_note.txt"},
    ),
]

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(documents)

['08fec27a-8edb-4b71-aff0-16ba22847fa8',
 'efb257ee-cae6-4734-8fda-92742f230d11',
 'be78fe32-d945-4922-81d4-3ecd62e15450',
 '555d1a58-5433-42ce-9a84-bb4674caac2c']

## 3. Research tools

Deep Agents accepts ordinary LangChain tools. We keep web search, fetch, semantic retrieval and keyword retrieval as custom tools.

File and shell tools are not recreated because the Deep Agents backend supplies them.

In [3]:
@tool
def web_search(query: str) -> str:
    """Search the live web using OpenAI's built-in web-search tool."""
    response = search_model.invoke(query, tools=[{"type": "web_search"}])

    lines = []
    for block in response.content_blocks:
        if block.get("type") == "text":
            lines.append(block.get("text", ""))
            for citation in block.get("annotations", []):
                if citation.get("url"):
                    lines.append(f"Source: {citation['url']}")
    return "\n".join(lines)


@tool
def fetch_url(url: str) -> str:
    """Fetch one web page and return readable text from it."""
    html = requests.get(url, timeout=20).text
    text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    return text[:12000]


@tool
def semantic_search(query: str) -> str:
    """Search the local teaching notes by meaning."""
    matches = vector_store.similarity_search(query, k=3)
    return "\n\n".join(
        f"{doc.metadata['source']}: {doc.page_content}"
        for doc in matches
    )


@tool
def keyword_search(query: str) -> str:
    """Search the local teaching notes using exact query words."""
    words = set(re.findall(r"\w+", query.lower()))
    scored = []

    for doc in documents:
        text_words = set(re.findall(r"\w+", doc.page_content.lower()))
        scored.append((len(words & text_words), doc))

    scored.sort(key=lambda item: item[0], reverse=True)

    return "\n\n".join(
        f"score={score} | {doc.metadata['source']}: {doc.page_content}"
        for score, doc in scored[:3]
    )

### Wrapping an existing function as a tool

`deep_agent_workspace/prices.py` already has a plain Python function, `get_price_series_yahoo`, that fetches a daily price history from Yahoo Finance via the `yfinance` package. Wrapping it with `@tool` is enough to make it callable by the agent -- we don't need to rewrite the underlying logic, just give the model a clear docstring and return a string it can read.

In [4]:
from deep_agent_workspace.prices import get_price_series_yahoo


@tool
def yahoo_price_series(ticker: str, period: str = "1y") -> str:
    """Look up a daily OHLCV price history for a stock ticker from Yahoo Finance.

    `period` is one of "1mo", "3mo", "6mo", "1y", "2y", "5y", "10y", "ytd", "max".
    Returns the oldest and most recent rows plus the row count, not the full series,
    to keep the tool's reply short enough for the agent to read easily.
    """
    try:
        rows = get_price_series_yahoo(ticker, period=period)
    except RuntimeError as exc:
        # Caught here, not left to raise, so a bad ticker (e.g. one not covered by
        # Yahoo Finance) becomes a normal tool result the agent can reason about,
        # instead of crashing the whole invoke() call.
        return f"{ticker.upper()}: lookup failed -- {exc}"

    first, last = rows[0], rows[-1]
    return (
        f"{ticker.upper()}: {len(rows)} trading days from {first['date']} to {last['date']}.\n"
        f"First close: {first['close']}\n"
        f"Last close: {last['close']}"
    )

## 4. Give Deep Agents a workspace

With this backend, Deep Agents can expose file operations such as `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob` and `grep`, plus `execute` for shell commands.

In [5]:
workspace = Path("deep_agent_workspace")
workspace.mkdir(exist_ok=True)

backend = LocalShellBackend(
    root_dir=str(workspace),
    virtual_mode=True,
)

## 5. Define specialist subagents

If `tools` is omitted, a custom subagent inherits the parent's custom tools.

The Manager delegates using Deep Agents' built-in `task` tool.

In [6]:
subagents = [
    {
        "name": "analyst",
        "description": (
            "Researches insurance and actuarial questions using web and local evidence. "
            "Use for fact finding, comparisons and numerical analysis."
        ),
        "system_prompt": (
            "You are the Analyst. Research carefully. Use web_search for current evidence, "
            "semantic_search for meaning-based local retrieval, keyword_search for exact terms, "
            "and fetch_url when you need one specific page. "
            "Return a concise evidence-based report with source URLs when available."
        ),
    },
    {
        "name": "reviewer",
        "description": (
            "Independently reviews an analysis for unsupported claims, missing risks, "
            "weak evidence and numerical inconsistencies."
        ),
        "system_prompt": (
            "You are the Reviewer. Challenge the work rather than merely summarising it. "
            "Use research tools when independent checking is useful. "
            "Return PASS or REVISE, followed by specific reasons."
        ),
    },
]

## 6. Create the Deep Agent Manager

`TodoListMiddleware` adds the optional `write_todos` planning tool.

In [7]:
deep_manager = create_deep_agent(
    model=model,
    tools=[web_search, fetch_url, semantic_search, keyword_search, yahoo_price_series],
    subagents=subagents,
    backend=backend,
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "You are the Manager. Use the analyst subagent for substantive research and "
        "the reviewer subagent to challenge important conclusions. "
        "For a multi-step task, use write_todos first. "
        "Use the workspace to save useful intermediate notes. "
        "For substantial answers, do not skip independent review."
    ),
)

## 7. First Deep Agents run

In [8]:
question = (
    "Prepare a short briefing on realistic generative-AI uses in insurance claims. "
    "Include benefits, failure modes and controls. Use current web evidence where useful. "
    "Save the final briefing as /final_briefing.md in the workspace."
)

result = deep_manager.invoke({
    "messages": [{"role": "user", "content": question}]
})

print(result["messages"][-1].text)

Created and saved the final briefing at:

`/final_briefing.md`

It includes realistic claims use cases, potential benefits, current web evidence, failure modes, practical controls, regulatory caveats, and a 90-day pilot approach.


## 8. Use the built-in filesystem tools

In [9]:
result = deep_manager.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "List the files in the workspace, then use grep to find occurrences "
            "of the word 'risk'. Briefly report what you found."
        ),
    }]
})

print(result["messages"][-1].text)

Files in the workspace:

- `/workspace/claims_genai_research_notes.md`

Occurrences of **“risk”** found: 4 lines (11, 13, 20, and 23). They refer to:

- Data privacy and security risks, including leakage, re-identification, vendor retention, and insecure logs.
- Bias and discrimination risks in claims decisions and fraud referrals.
- A citation to NIST’s Generative AI Risk Management Framework.
- A citation to Swiss Re’s claims-related risk research.


## 9. Demonstrate shell execution

Keep the classroom example harmless.

In [10]:
result = deep_manager.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Use the execute tool to run 'python --version' and 'pwd'. "
            "Tell me what they show."
        ),
    }]
})

print(result["messages"][-1].text)

- `python --version`: `python` was not found (`/bin/sh: 1: python: not found`)
- `pwd`: `/home/nodozi/projects/Intern_GenAI_Examples/deep_agent_workspace`


## 10. Parallel subagents

A Deep Agent can issue several `task` calls in one turn, allowing delegated work to run in parallel.

In [11]:
result = deep_manager.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Investigate these two questions independently and in parallel where possible: "
            "(1) benefits of generative AI in claims operations, "
            "(2) privacy and governance risks. "
            "Use the analyst for the first and the reviewer as an independent risk researcher "
            "for the second, then combine the findings."
        ),
    }]
})

print(result["messages"][-1].text)

## Executive summary

Generative AI has a credible near-term role in claims operations, especially as an **adjuster and operations copilot**. The strongest benefits are faster handling of unstructured information, reduced documentation effort, improved communications, and better routing of claims.

The most serious risks arise because claims contain highly sensitive personal information and because AI outputs can influence coverage, payment, fraud, and settlement decisions. The appropriate posture is **human-supervised augmentation**, with strict data controls and auditable decision processes—not unrestricted autonomous adjudication.

---

# 1. Benefits of generative AI in claims operations

## Highest-value use cases

| Claims activity | Potential GenAI application | Benefit strength |
|---|---|---|
| **FNOL and intake** | Conversational intake, transcription, extraction of facts from calls and forms, identifying missing information, creating structured claim records | High |
| **Docu

## 11. Inspect the run

In [12]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Investigate these two questions independently and in parallel where possible: (1) benefits of generative AI in claims operations, (2) privacy and governance risks. Use the analyst for the first and the reviewer as an independent risk researcher for the second, then combine the findings.
================================== Ai Message ==================================

[{'id': 'rs_0366db0edf2095c3006a9d469c625887d2b7adc39f2ccef42e', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqnUadhtm2KPcWg3fXT_6B21ptrlooAzXaqjWKCjPSjWbHcm7pwgyOULzXibVZDIV9QhLZTUITs7lH7D_bOAvPu-1qwo1AubeNTiICOUPX8Zfjq68AeeM_jWGdgTCQt82LFWeUSzcJpFXG40y2bP-RwrVslP7uDbB_G4adCbCIE_7G7e2dmHdBBsbgaDy5XwDKtqMzflmE5QqxQ6Fhlz88OYEk2wGZJBIuiGogJE4fAx9oMUkrMwGh36QzICRtmhdpnOda-OMSJPGESpvJuRA7OCe1qMzh49s1uO_y6hO8nGUCFHl9G4L4NJefGk7g2rrz-AUFyNDR4diBEV_gkFZBwvW4d4xVoMC-o_jgEGuVaGHbwYw4qwIAffy9V6MKSEny1aCgxs3cote6YUOMoX

## 12. Exercise -- delegate price lookups, with a deliberate failure

Ask the Manager to compare four insurers/companies: **Legal & General** (`LGEN.L`), **Aviva** (`AV.L`), **AIICO Insurance** (`AIICO.LG`), and **MTN Group** (`MTN.JO`). AIICO trades on the Nigerian Exchange, which Yahoo Finance does not cover -- that lookup will fail. MTN Group trades on the Johannesburg Stock Exchange, which Yahoo Finance does cover -- that one succeeds. Having one failure and one success from Africa side by side means the failure is really about exchange coverage, not "Yahoo Finance doesn't do Africa."

Before running the next cell, predict:

1. Will the four `yahoo_price_series` calls happen in one turn (parallel) or across several turns (sequential)?
2. What will the analyst report back about AIICO once its lookup fails?
3. Will the reviewer catch that one of the four lookups is missing, or let it slide?

In [ ]:
question = (
    "Compare the 1-year share price performance of four companies: "
    "Legal & General (LGEN.L), Aviva (AV.L), AIICO Insurance (AIICO.LG), "
    "and MTN Group (MTN.JO). "
    "Delegate the price lookups to the analyst subagent, running the lookups "
    "in parallel where possible. If a lookup fails, note that clearly rather than "
    "guessing a number. Then send the analyst's draft to the reviewer subagent "
    "to check the numbers are consistent and that any failed lookup was flagged, "
    "not silently dropped. Save the final comparison as /price_comparison.md."
)

result = deep_manager.invoke({
    "messages": [{"role": "user", "content": question}]
})

print(result["messages"][-1].text)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AIICO.LG"}}}
$AIICO.LG: No data found, symbol may be delisted


Check your prediction from step 1: look for `task` tool calls in the message trace. If more than one appears on the same AI message (rather than spread across separate messages), those subagent runs were requested in parallel.

In [ ]:
import json
from pathlib import Path

log_path = Path("deep_agent_workspace/logs/price_lookups.jsonl")
entries = [json.loads(line) for line in log_path.read_text().splitlines()]

for entry in entries[-4:]:
    print(entry["ticker"], entry["status"], entry.get("error"))

## Why this feels different from Notebook 1

**Notebook 1:** we created file tools, shell execution and agent-as-tool delegation ourselves.

**Notebook 2:** Deep Agents supplies a harness around the same core LangChain ideas.

### Common gotchas

- Deep Agents subagents have fresh context; they do not continuously chat with one another.
- A subagent returns a handoff to the parent.
- `LocalShellBackend` is convenient, but it is not isolated from your computer.
- More tools are not always better.
- Planning is opt-in, so we explicitly add `TodoListMiddleware`.